# Surrogate Modelling Workshop Exercise

This is the participant workbook for the workshop.
The tutorial notebook remains the facilitator cheat sheet and answer key.

## How to use this notebook

1. Run the cells from top to bottom.
2. Only edit lines marked `TODO`.
3. Keep the edits small: usually one number, one short list, or one pair of indices.
4. After each section, write a short engineering conclusion, not a Python explanation.

Target working time: about 30 to 40 minutes in pairs.


## Scenario

You are given a synthetic stand-in for an expensive structural simulation with three design variables:
- `t`: sheet thickness in mm,
- `sigma_y`: yield strength in MPa,
- `r`: bead radius in mm.

The output is `intrusion` in mm, where lower is better.
Pretend each call to `mechanical_response()` is a costly FE run, so every extra sample must be justified.

The exercise is not about writing much code. It is about making reasonable engineering decisions from a small surrogate-modelling workflow.


## Deliverable

By the end of the exercise, each team should be able to give a short recommendation covering:
- the DOE size you chose,
- which surrogate you would trust most for screening,
- what evidence supports that choice,
- which variables seem most influential near the nominal design,
- where you would want a few extra simulations before relying on the model.

A good answer is a short engineering argument backed by plots and metrics.


In [ ]:
# If you run this notebook in Colab, uncomment the next line.
# %pip install smt

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error, r2_score

from smt.sampling_methods import LHS
from smt.surrogate_models import LS, QP, KRG
from smt.utils.misc import compute_q2, compute_rmse


In [ ]:
# Design variable bounds: [min, max]
xlimits = np.array([
    [0.8, 2.0],     # thickness t [mm]
    [180.0, 420.0], # yield strength sigma_y [MPa]
    [3.0, 12.0],    # bead radius r [mm]
])

var_names = ["thickness t [mm]", "yield strength sigma_y [MPa]", "bead radius r [mm]"]
pair_indices = [(0, 1), (0, 2), (1, 2)]
x_nominal = xlimits.mean(axis=1)


def mechanical_response(x):
    """Synthetic stand-in for an expensive mechanical simulation."""
    x = np.atleast_2d(x)
    t = x[:, 0]
    sigma_y = x[:, 1]
    r = x[:, 2]

    intrusion = (
        27.0
        - 8.5 * t
        - 0.018 * sigma_y
        + 0.55 * r
        + 3.0 * (t - 1.35) ** 2
        + 0.000035 * (sigma_y - 300.0) ** 2
        + 0.09 * (r - 7.0) ** 2
        - 0.010 * t * (sigma_y - 300.0)
        - 0.12 * t * r
        + 0.0025 * r * (sigma_y - 300.0)
        + 0.7 * np.sin(0.9 * r)
    )
    return intrusion.reshape(-1, 1)


def make_pair_grid(ix, iy, fixed=None, n=50):
    fixed = x_nominal.copy() if fixed is None else np.array(fixed, dtype=float).copy()
    xi = np.linspace(xlimits[ix, 0], xlimits[ix, 1], n)
    yi = np.linspace(xlimits[iy, 0], xlimits[iy, 1], n)
    X, Y = np.meshgrid(xi, yi)
    pts = np.tile(fixed, (n * n, 1))
    pts[:, ix] = X.ravel()
    pts[:, iy] = Y.ravel()
    return X, Y, pts


## Task 1: Choose a DOE size and generate the data

Change only one main input in the next cell: `ndoe`.

Start with `30` unless you have a reason to spend more or fewer solver calls.
The rest of the cell is already prepared so the team can focus on the tradeoff, not the syntax.

Questions to answer after you run it:
- Is your DOE size plausible if each run were expensive?
- Would you spend more or fewer solver calls on training for this problem?


In [ ]:
# TODO: change this number if you want to test a different training budget
ndoe = 30

# Keep the validation set fixed so teams compare models on the same basis
nval = 150

sampling_train = LHS(xlimits=xlimits, criterion="ese", seed=7)
sampling_val = LHS(xlimits=xlimits, criterion="ese", seed=21)

xt = sampling_train(ndoe)
yt = mechanical_response(xt)

xval = sampling_val(nval)
yval = mechanical_response(xval)

print(f"Training samples: {len(xt)}")
print(f"Validation samples: {len(xval)}")
print(f"Intrusion range in training set: {yt.min():.2f} to {yt.max():.2f} mm")

## Task 2: Check whether the DOE looks trustworthy

Run the next cell and inspect the point coverage.
If you want, change `view_pairs` to inspect a different pair of variables.

Questions:
- Does the DOE cover the space reasonably well for a first surrogate?
- Are there obvious empty regions or strong clusters?
- Would you trust this sampling plan more than a purely random one?


In [ ]:
# TODO: change one pair if you want to inspect a different view
view_pairs = [(0, 1), (0, 2)]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (ix, iy) in zip(axes, view_pairs):
    ax.scatter(xt[:, ix], xt[:, iy], c=yt[:, 0], cmap="viridis", s=70)
    ax.set_xlabel(var_names[ix])
    ax.set_ylabel(var_names[iy])
    ax.set_title(f"DOE colored by intrusion: {var_names[ix]} vs {var_names[iy]}")

plt.tight_layout()
plt.show()

### Team Note: DOE Choice

Write 2 or 3 short lines here after Task 2.

- Does your DOE look good enough for a first surrogate?
- If not, what would you change first: more points, different placement, or both?

## Task 3: Look at the true response before fitting models

Run the next cell to inspect the true response on a couple of 2D slices.
If you want, change `slice_pairs` to inspect the third pair of variables.

Questions:
- Which variables seem to show the strongest curvature?
- Where do you see signs of interaction rather than a simple linear trend?
- Based on these plots, would a purely linear surrogate be enough?


In [ ]:
# TODO: change one slice if you want to inspect another pair
slice_pairs = [(0, 1), (0, 2)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (ix, iy) in zip(axes, slice_pairs):
    X, Y, pts = make_pair_grid(ix, iy, n=60)
    Z = mechanical_response(pts).reshape(X.shape)

    contour = ax.contourf(X, Y, Z, levels=18, cmap="viridis")
    ax.contour(X, Y, Z, levels=10, colors="white", linewidths=0.5, alpha=0.7)
    ax.scatter(xt[:, ix], xt[:, iy], c="black", s=16, alpha=0.6)
    ax.set_xlabel(var_names[ix])
    ax.set_ylabel(var_names[iy])
    fixed_idx = [k for k in range(3) if k not in (ix, iy)][0]
    ax.set_title(f"{var_names[ix]} vs {var_names[iy]}\n{var_names[fixed_idx]} fixed at nominal")

fig.colorbar(contour, ax=axes, shrink=0.9, pad=0.02, label="intrusion [mm]")
plt.tight_layout()
plt.show()

## Task 4: Train a short list of surrogate models, pick the best one

Do not write the training loop yourself.
In the next cell, only change:
- `selected_model_names` if you want to compare fewer models,
- optionally `ranking_metric` if you want to discuss a different selection rule.

Questions:
- Which model would you pick for screening?
- Is the winner clearly better, or only slightly better?
- If this were a real FE campaign, would the difference matter enough to change your decision?


In [ ]:
selected_model_names = ["LS", "QP", "KRG"]

ranking_metric = "val_rmse"

model_bank = {
    "LS": LS(print_prediction=False),
    "QP": QP(print_prediction=False),
    "KRG": KRG(theta0=[1e-2] * xlimits.shape[0], print_prediction=False),
}

results = []
trained_models = {}

for name in selected_model_names:
    model = model_bank[name]
    model.set_training_values(xt, yt)
    model.train()

    yhat_train = model.predict_values(xt)
    yhat_val = model.predict_values(xval)

    results.append({
        "model": name,
        "train_rmse": float(np.sqrt(mean_squared_error(yt, yhat_train))),
        "val_rmse": float(np.sqrt(mean_squared_error(yval, yhat_val))),
        "train_r2": float(r2_score(yt, yhat_train)),
        "val_r2": float(r2_score(yval, yhat_val)),
        "q2": float(compute_q2(model, xval, yval)),
    })
    trained_models[name] = model


In [ ]:
results = sorted(results, key=lambda d: d[ranking_metric])

for row in results:
    print(
        f"{row['model']:>3} | train RMSE = {row['train_rmse']:.3f} | val RMSE = {row['val_rmse']:.3f} | "
        f"train R2 = {row['train_r2']:.3f} | val R2 = {row['val_r2']:.3f} | Q2 = {row['q2']:.3f}"
    )

### Model Choice

Review the above output, which model performs best?

- Which model did you choose?
- Which metric or visual cue mattered most for that choice?

In [ ]:
# TODO: Pick the best model, choose between ["LS", "QP", "KRG"]
# TODO: Pick another model later if you want to test it out
best_name = "KRG"
best_model = trained_models[best_name]

print()
print(f"Selected best model using {ranking_metric}: {best_name}")

## Task 5: Check whether the selected surrogate is trustworthy

Run the next cell.
In this exercise we can plot absolute error because the synthetic response is known everywhere.
In a real simulation campaign that is usually not possible, so the most useful substitute is a model-based uncertainty estimate when the selected surrogate provides one.

It creates a validation parity plot, an absolute-error contour map, and, for Kriging-type models, a predictive-uncertainty map for one slice of the design space.

If you want, change `ix, iy` in the next cell to inspect a different pair of variables.

Questions:
- Does the model look reliable everywhere, or only in some regions?
- Where would you be cautious before using the surrogate for decisions?
- If you had budget for 5 more FE runs, where would you place them?
- If uncertainty is available, does the high-uncertainty region match where you would add more simulations?
- Would you treat uncertainty here as a guide for new samples, or as the true error?

In [ ]:
best_pred = best_model.predict_values(xval)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(yval, yval, "k-", label="ideal")
axes[0].scatter(yval, best_pred, color="tab:red", alpha=0.7, label=best_name)
axes[0].set_xlabel("True intrusion [mm]")
axes[0].set_ylabel("Predicted intrusion [mm]")
axes[0].set_title(f"Validation parity plot: {best_name}")
axes[0].legend()
axes[0].grid(alpha=0.3)

# TODO: change this pair if you want to inspect another error map
# 0 = thickness, 1 = yield strength, 2 = bead radius
ix, iy = 0, 1
X, Y, pts = make_pair_grid(ix, iy, n=70)
Z_true = mechanical_response(pts).reshape(X.shape)
Z_pred = best_model.predict_values(pts).reshape(X.shape)
Z_err = np.abs(Z_true - Z_pred)

error_map = axes[1].contourf(X, Y, Z_err, levels=20, cmap="magma")
axes[1].scatter(xt[:, ix], xt[:, iy], c="white", s=12, alpha=0.7)
axes[1].set_title(f"Absolute error: {var_names[ix]} vs {var_names[iy]}")
axes[1].set_xlabel(var_names[ix])
axes[1].set_ylabel(var_names[iy])
fig.colorbar(error_map, ax=axes[1], shrink=0.85, label="abs error [mm]")

if hasattr(best_model, "predict_variances"):
    Z_var = best_model.predict_variances(pts).reshape(X.shape)
    Z_std = np.sqrt(np.maximum(Z_var, 0.0))

    uncertainty_map = axes[2].contourf(X, Y, Z_std, levels=20, cmap="cividis")
    axes[2].scatter(xt[:, ix], xt[:, iy], c="white", s=12, alpha=0.7)

    peak_idx = np.unravel_index(np.argmax(Z_std), Z_std.shape)
    x_peak = X[peak_idx]
    y_peak = Y[peak_idx]
    z_peak = Z_std[peak_idx]

    axes[2].scatter(x_peak, y_peak, color="tab:red", marker="x", s=80, label="highest uncertainty")
    axes[2].set_title(f"Predictive std: {var_names[ix]} vs {var_names[iy]}")
    axes[2].set_xlabel(var_names[ix])
    axes[2].set_ylabel(var_names[iy])
    axes[2].legend()
    fig.colorbar(uncertainty_map, ax=axes[2], shrink=0.85, label="predicted std [mm]")

    print(
        "Predictive uncertainty is a model-based guide for where more samples may help; "
        "it is not the true error."
    )
    print(
        f"Highest predicted uncertainty on this slice near {var_names[ix]} = {x_peak:.3f}, "
        f"{var_names[iy]} = {y_peak:.3f} with std = {z_peak:.3f} mm"
    )
else:
    axes[2].text(
        0.5,
        0.5,
        f"Predictive uncertainty is not available\nfor {best_name}. Use validation error\nand engineering judgment instead.",
        ha="center",
        va="center",
        fontsize=12,
    )
    axes[2].set_title("Predictive uncertainty not available")
    axes[2].set_axis_off()

plt.tight_layout()
plt.show()

## Task 6: Quick sensitivity check near the nominal design

Run the next cell.
The goal is not a full sensitivity study, only a fast local view of which variables matter most near the nominal design.

Question:
- Which variable seems most influential near the nominal point?
- Is that consistent with the response-surface plots you saw earlier?


In [ ]:
base = x_nominal.copy()
base_y = mechanical_response(base)[0, 0]
normalized_sensitivities = []

# TODO: keep this as a small fraction of the design range
step_fraction = 0.02

for i in range(3):
    dx = step_fraction * (xlimits[i, 1] - xlimits[i, 0])
    x_plus = base.copy()
    x_minus = base.copy()
    x_plus[i] += dx
    x_minus[i] -= dx
    y_plus = mechanical_response(x_plus)[0, 0]
    y_minus = mechanical_response(x_minus)[0, 0]
    slope = (y_plus - y_minus) / (2.0 * dx)
    normalized_sensitivities.append(slope * base[i] / base_y)

plt.figure(figsize=(7, 4))
plt.bar(["t", "sigma_y", "r"], normalized_sensitivities, color=["tab:blue", "tab:orange", "tab:green"])
plt.axhline(0.0, color="black", linewidth=1)
plt.ylabel("normalized local sensitivity")
plt.title("Normalized local sensitivity at nominal design")
plt.grid(axis="y", alpha=0.3)
plt.show()

for name, value in zip(["t", "sigma_y", "r"], normalized_sensitivities):
    print(f"{name}: {value:.3f}")

## Task 7: Use the surrogate for fast screening

Run the next cell to screen a much larger candidate set with the selected surrogate.
Then compare the top-ranked designs against the true response.

Questions:
- Is the surrogate good enough for pre-screening?
- Would you trust it for final decisions, or only to narrow the candidate list?


In [ ]:
# TODO: increase or decrease this if you want to test a larger screening sweep
n_screen = 5000

sampling_screen = LHS(xlimits=xlimits, criterion="ese", seed=101)
x_screen = sampling_screen(n_screen)

y_screen_pred = best_model.predict_values(x_screen)
rank = np.argsort(y_screen_pred[:, 0])

x_top = x_screen[rank[:10]]
y_top = y_screen_pred[rank[:10]]
y_top_true = mechanical_response(x_top)

print("Top 10 surrogate-ranked candidates")
print("t [mm]   sigma_y [MPa]   r [mm]   predicted   true   abs error")
for x_i, yp, yt_i in zip(x_top, y_top[:, 0], y_top_true[:, 0]):
    print(
        f"{x_i[0]:6.3f}   {x_i[1]:14.1f}   {x_i[2]:6.3f}   "
        f"{yp:9.3f} {yt_i:7.3f} {abs(yp - yt_i):10.3f}"
    )

print()
print(f"RMSE on the top-10 verification set: {compute_rmse(best_model, x_top, y_top_true):.3f}")

## Final Debrief

If you have time, compare your recommendation against another team and look for differences in:
- DOE size choice,
- surrogate selection,
- how much visual validation mattered versus metrics,
- whether teams trusted the model only for screening or for more than that.

The point is not to get identical answers.
The point is to make a defensible engineering recommendation from limited data.


## Optional Extensions

Only do these if your team finishes early.
Each one should require only a small change to an existing cell.

1. Increase `ndoe` from 30 to 60. Which model benefits most?
2. Remove one input variable and see whether `QP` becomes closer to `KRG`.
3. Add random noise to `yt` to mimic noisy simulations. Does the preferred model change?
4. Compare your good DOE with a deliberately clustered DOE.
5. Replace the synthetic response with one of your own simulation datasets.
